# Azure AI Search Backend — Managed Cloud Vector Search

Medha 0.3.0 ships with `AzureSearchBackend`: a storage backend that uses
**Azure AI Search** (formerly Azure Cognitive Search) for vector similarity
search. Azure AI Search provides a fully managed HNSW index, OData-style
payload filtering, and automatic scaling — no infrastructure to operate.

This notebook covers:
1. **API key authentication** — fastest setup
2. **Store & search** — full waterfall on Azure AI Search
3. **AAD authentication** — `DefaultAzureCredential` (placeholder)
4. **Tuning** — `azure_search_top_k_candidates`
5. **TTL + `expire()`** — time-limited entries
6. **Multi-index** — per-collection index mapping
7. **`invalidate_collection()`** — drop index and cleanup
8. **Elasticsearch vs Azure AI Search** comparison

**Requirements:**
```bash
pip install "medha-archai[azure-search,fastembed]"
```

**Cloud prerequisites:**
1. Create an Azure AI Search service in the Azure portal
2. Copy the **endpoint** and **API key** from *Settings → Keys*

## Prerequisites

```bash
pip install "medha-archai[azure-search,fastembed]"
# optional: pip install azure-identity  (for AAD auth)
```

Set environment variables:
```bash
export MEDHA_AZURE_SEARCH_ENDPOINT="https://my-service.search.windows.net"
export MEDHA_AZURE_SEARCH_API_KEY="your-api-key"
```

In [ ]:
import asyncio
import os
import time

from medha import Medha, Settings
from medha.embeddings.fastembed_adapter import FastEmbedAdapter

try:
    from medha import AzureSearchBackend
    HAS_AZURE = True
    print("azure-search-documents package is installed")
except ImportError:
    HAS_AZURE = False
    print("azure-search-documents not found — install with: pip install medha-archai[azure-search]")

AZURE_ENDPOINT = os.environ.get("MEDHA_AZURE_SEARCH_ENDPOINT", "")
AZURE_API_KEY  = os.environ.get("MEDHA_AZURE_SEARCH_API_KEY", "")

if AZURE_ENDPOINT and AZURE_API_KEY:
    print(f"Endpoint : {AZURE_ENDPOINT}")
    print(f"API key  : ...{AZURE_API_KEY[-4:]}")
else:
    print("MEDHA_AZURE_SEARCH_ENDPOINT / MEDHA_AZURE_SEARCH_API_KEY not set")
    print("Cells requiring Azure AI Search will be skipped.")

CAN_RUN = HAS_AZURE and bool(AZURE_ENDPOINT) and bool(AZURE_API_KEY)

embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")

## Cell 1: Authentication with API Key

In [ ]:
if not CAN_RUN:
    print("Skipping — Azure AI Search not configured.")
else:
    settings = Settings(
        backend_type="azure-search",
        azure_search_endpoint=AZURE_ENDPOINT,
        azure_search_api_key=AZURE_API_KEY,
        azure_search_index_name="medha-demo",
    )

    print(f"backend_type              : {settings.backend_type}")
    print(f"azure_search_endpoint     : {settings.azure_search_endpoint}")
    print(f"azure_search_index_name   : {settings.azure_search_index_name}")
    print(f"azure_search_top_k_cand   : {settings.azure_search_top_k_candidates}")

    async with Medha("az_demo", embedder=embedder, settings=settings) as medha:
        print(f"\nBackend: {type(medha._backend).__name__}")

## Cell 2: Store 3 Entries + Search + Print CacheHit

In [ ]:
if not CAN_RUN:
    print("Skipping — Azure AI Search not configured.")
else:
    settings = Settings(
        backend_type="azure-search",
        azure_search_endpoint=AZURE_ENDPOINT,
        azure_search_api_key=AZURE_API_KEY,
        azure_search_index_name="medha-demo",
    )

    async with Medha("az_demo", embedder=embedder, settings=settings) as medha:
        await medha.store("How many users are registered?",
                          "SELECT COUNT(*) FROM users")
        await medha.store("List active sessions",
                          "SELECT * FROM sessions WHERE active = true")
        await medha.store("Monthly revenue total",
                          "SELECT SUM(amount) FROM orders WHERE MONTH(created_at)=MONTH(NOW())")

        hit = await medha.search("Total number of registered users")
        print(f"strategy   : {hit.strategy.value}")
        print(f"confidence : {hit.confidence:.4f}")
        print(f"query      : {hit.generated_query}")

## Cell 3: AAD Authentication — `DefaultAzureCredential` (Placeholder)

When `azure_search_api_key` is `None`, Medha uses `DefaultAzureCredential`
from the `azure-identity` package. This supports managed identities, service
principals, and developer CLI credentials.

**Prerequisites:**
- `pip install azure-identity`
- Assign the **"Search Index Data Contributor"** role to the identity

In [ ]:
# ⚠️ Placeholder — do not execute without real AAD credentials
settings_aad = Settings(
    backend_type="azure-search",
    azure_search_endpoint="https://my-service.search.windows.net",
    azure_search_api_key=None,   # triggers DefaultAzureCredential
    azure_search_index_name="medha-prod",
)

print(f"api_key set : {settings_aad.azure_search_api_key is not None}")
print("Medha will use DefaultAzureCredential (requires azure-identity + RBAC role)")
print("(Not connecting — placeholder values)")

## Cell 4: Tuning — `azure_search_top_k_candidates`

`azure_search_top_k_candidates` controls how many extra candidates the HNSW
index fetches before score filtering. Higher values improve recall at the cost
of latency and cost.

In [ ]:
if not CAN_RUN:
    print("Skipping — Azure AI Search not configured.")
else:
    test_question = "How many users signed up?"

    for top_k in [10, 50, 200]:
        settings = Settings(
            backend_type="azure-search",
            azure_search_endpoint=AZURE_ENDPOINT,
            azure_search_api_key=AZURE_API_KEY,
            azure_search_index_name="medha-demo",
            azure_search_top_k_candidates=top_k,
        )
        async with Medha("az_tuning", embedder=embedder, settings=settings) as medha:
            await medha.store("How many users are registered?", "SELECT COUNT(*) FROM users")
            t0 = time.perf_counter()
            hit = await medha.search(test_question)
            elapsed = (time.perf_counter() - t0) * 1000
            print(
                f"  top_k_candidates={top_k:>4d}  "
                f"strategy={hit.strategy.value:<18s}  "
                f"conf={hit.confidence:.4f}  "
                f"{elapsed:.1f}ms"
            )

## Cell 5: TTL + `expire()` — Time-Limited Entries

Expiry is stored as a `DateTimeOffset` field in the Azure Search document.
`expire()` deletes documents where `expires_at` is in the past.

In [ ]:
if not CAN_RUN:
    print("Skipping — Azure AI Search not configured.")
else:
    settings = Settings(
        backend_type="azure-search",
        azure_search_endpoint=AZURE_ENDPOINT,
        azure_search_api_key=AZURE_API_KEY,
        azure_search_index_name="medha-demo",
    )

    async with Medha("az_ttl", embedder=embedder, settings=settings) as medha:
        await medha.store(
            "Today's total revenue",
            "SELECT SUM(amount) FROM orders WHERE DATE(created_at) = CURDATE()",
            ttl=3600,   # expires_at stored as DateTimeOffset in Azure Search document
        )
        await medha.store(
            "Current time query",
            "SELECT NOW()",
            ttl=1,      # expires almost immediately
        )

        await asyncio.sleep(2)
        removed = await medha.expire()
        print(f"Removed {removed} expired entries")

## Cell 6: Multi-Index — Per-Collection Index Mapping

Azure AI Search index names are derived from `azure_search_index_name` + `collection_name`.
Different collections map to different indexes, enabling environment or tenant isolation.

In [ ]:
# Demonstrate the naming convention (no connection required)
for index_prefix, collection in [
    ("medha-prod",    "sql-cache"),
    ("medha-prod",    "cypher-cache"),
    ("medha-staging", "sql-cache"),
]:
    s = Settings(
        backend_type="azure-search",
        azure_search_index_name=index_prefix,
    )
    # Final Azure index = "{index_name}-{collection_name}" (hyphens, lowercased)
    az_index = f"{index_prefix}-{collection.replace('_', '-')}"
    print(f"  index_name={index_prefix!r:20s} collection={collection!r:16s} → Azure index: {az_index!r}")

## Cell 7: `invalidate_collection()` — Drop Index

In [ ]:
if not CAN_RUN:
    print("Skipping — Azure AI Search not configured.")
else:
    settings = Settings(
        backend_type="azure-search",
        azure_search_endpoint=AZURE_ENDPOINT,
        azure_search_api_key=AZURE_API_KEY,
        azure_search_index_name="medha-demo",
    )

    async with Medha("az_demo", embedder=embedder, settings=settings) as medha:
        n = await medha.invalidate_collection()
        print(f"invalidate_collection() removed {n} entries (index dropped and recreated)")

## Elasticsearch vs Azure AI Search

| Feature | Elasticsearch | Azure AI Search |
|---|---|---|
| **Deployment** | Self-hosted or Elastic Cloud | Fully managed (Azure) |
| **Scaling** | Manual sharding | Automatic |
| **Filter syntax** | Lucene / DSL | OData (`$filter`) |
| **Vector index** | HNSW | HNSW (managed) |
| **Auth** | API key / Basic / OIDC | API key / AAD / Managed Identity |
| **Cost model** | Self-hosted infra | Pay-per-unit (SU) |
| **Best for** | Elastic stack teams | Azure-first teams, zero-ops |

**Choose Azure AI Search** when:
- Azure is already your cloud provider
- You want zero infrastructure operations
- Your dataset exceeds 1M entries and needs automatic scaling
- You need AAD / Managed Identity for compliance